# Data Reliability - 事故处理 (Incident Handling)

> **适用场景**: 数据 pipeline 故障处理、生产事故响应
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频（Senior DE 必考）

## 目录
1. Pipeline Failure RCA（根因分析）
2. Rollback & Replay Strategy
3. Backfill 安全执行
4. On-call Runbook
5. 练习题

---
## 1. Pipeline Failure RCA（根因分析）

### RCA 流程
```
检测 (Detect)
    ↓
分类 (Triage) — 确定影响范围和优先级
    ↓
临时缓解 (Mitigate) — 止血，恢复服务
    ↓
根因分析 (Root Cause) — 找到真正原因
    ↓
修复 (Fix) — 永久解决
    ↓
复盘 (Post-mortem) — 防止复发
```

### 常见故障根因分类

| 根因类别 | 典型症状 | 排查方向 |
|----------|----------|----------|
| **上游数据问题** | 今天行数比昨天少70% | 检查源表/API/Kafka offset |
| **Schema 变更** | Pipeline 跑到一半报列不存在 | 检查源表 DDL 历史 |
| **资源不足** | OOM、超时、Task 被 kill | 检查内存/CPU/磁盘使用 |
| **依赖超时** | Sensor 等了太久上游 | 检查上游 pipeline 状态 |
| **代码 Bug** | 新发布后立刻失败 | 对比代码变更 diff |
| **基础设施故障** | 云服务不可用 | 查云厂商 Status Page |
| **数据倾斜/热点** | 某个分区处理特别慢 | 检查数据分布 |

### RCA 排查步骤（实战）
```bash
# 1. 确认故障范围
# - 哪个 DAG/Task 失败？
# - 什么时间开始失败？
# - 是第一次失败还是反复失败？

# 2. 查看错误日志
# Airflow UI → DAG → Task → Logs
# 关键信息：错误类型、堆栈 trace、失败时间

# 3. 检查上游状态
SELECT COUNT(*), MAX(created_at)
FROM raw.orders
WHERE DATE(created_at) = CURRENT_DATE;
-- 若为 0 → 上游未推送数据

# 4. 检查最近变更
git log --since='2024-01-15' --oneline  # 最近代码变更
# 数仓 DDL 历史
SELECT * FROM INFORMATION_SCHEMA.TABLE_CHANGES
WHERE table_name = 'orders' ORDER BY timestamp DESC LIMIT 10;

# 5. 5 Whys 分析
# Why 1: fct_orders 今天没有数据
# Why 2: staging.stg_orders 为空
# Why 3: raw.orders 今天分区为空
# Why 4: Kafka consumer 没有消费到数据
# Why 5: Kafka broker 磁盘满了，停止写入
# → 根因：Kafka 磁盘告警阈值设置过高，没有提前预警
```

---
## 2. Rollback & Replay Strategy

### 何时需要 Rollback？
- 新代码发布后数据计算逻辑错误
- 上游数据污染（Bad data 已写入仓库）
- Schema 变更导致数据错误

### Rollback 策略

**策略 1：分区覆盖（最常用）**
```sql
-- 用正确逻辑重新计算，覆盖有问题的分区
-- BigQuery 分区覆盖
INSERT OVERWRITE TABLE analytics.fct_orders
PARTITION (date = '2024-01-15')
SELECT ... FROM staging.stg_orders WHERE DATE(order_time) = '2024-01-15';

-- Spark
df.write \
    .mode('overwrite') \
    .option('partitionOverwriteMode', 'dynamic') \
    .partitionBy('date') \
    .parquet('/path/to/table/')
```

**策略 2：Delta Lake Time Travel**
```sql
-- 查看历史版本
DESCRIBE HISTORY delta.`/path/to/orders`;

-- 回滚到特定版本
RESTORE TABLE delta.`/path/to/orders` TO VERSION AS OF 10;

-- 或按时间点
RESTORE TABLE orders TO TIMESTAMP AS OF '2024-01-14 23:00:00';
```

**策略 3：代码回滚 + 重跑**
```bash
# 1. 回滚代码到上一个正确版本
git revert HEAD  # 或 git checkout v1.2.3

# 2. 重新部署
dbt run --target prod --select fct_orders+ --full-refresh

# 3. 通过 Airflow 手动触发指定日期的 backfill
airflow dags backfill orders_pipeline -s 2024-01-15 -e 2024-01-15
```

### Replay（数据重放）
```python
# Kafka 重放：从指定 offset 重新消费
consumer = KafkaConsumer(
    'orders-topic',
    bootstrap_servers=['kafka:9092'],
    auto_offset_reset='earliest',  # 或指定具体 offset
)

# 手动设置 offset（回放特定时间段）
from kafka import TopicPartition
tp = TopicPartition('orders-topic', 0)
consumer.assign([tp])
consumer.seek(tp, offset=1000)  # 从 offset 1000 开始
```

---
## 3. Backfill 安全执行

### Backfill 的风险
- **重复数据**：非幂等 pipeline 重跑产生重复记录
- **资源占用**：大范围 backfill 占满计算资源，影响实时 pipeline
- **下游影响**：Backfill 期间下游看到不完整/不正确的中间状态
- **成本飙升**：全量重算代价极高

### 安全 Backfill 步骤

**步骤 1：评估影响范围**
```python
# 确认需要重跑的时间范围
# - 问题从什么时候开始？
# - 影响哪些下游表？（用血缘图）
# - 多少数据量？

SELECT COUNT(*), MIN(date), MAX(date)
FROM fct_orders
WHERE date BETWEEN '2024-01-10' AND '2024-01-15';
```

**步骤 2：保证幂等性**
```sql
-- 方案 A：分区覆盖（最安全）
-- 每次重算某一天的数据，完整覆盖该分区

-- 方案 B：MERGE（upsert）
MERGE INTO fct_orders AS target
USING new_data AS source ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET ...
WHEN NOT MATCHED THEN INSERT ...;

-- 方案 C：先删后插
DELETE FROM fct_orders WHERE date = '2024-01-15';
INSERT INTO fct_orders SELECT ... WHERE date = '2024-01-15';
-- （需要在事务中执行）
```

**步骤 3：控制执行速度**
```python
# Airflow backfill 限制并发
airflow dags backfill orders_pipeline \
    -s 2024-01-10 \
    -e 2024-01-15 \
    --max-active-runs 1  # 一次只跑一天，避免资源竞争

# 在非高峰时段执行（如深夜）
# 或降低 Spark executor 数量，给实时 pipeline 留资源
```

**步骤 4：验证结果**
```sql
-- Backfill 完成后验证
-- 1. 行数对账
SELECT date, COUNT(*) FROM fct_orders
WHERE date BETWEEN '2024-01-10' AND '2024-01-15'
GROUP BY 1 ORDER BY 1;

-- 2. 与源系统对账
SELECT
    s.date,
    s.cnt AS source_cnt,
    t.cnt AS target_cnt,
    s.cnt = t.cnt AS match
FROM source_counts s
JOIN target_counts t USING (date);

-- 3. 检查关键指标是否合理（与历史同期对比）
```

**步骤 5：通知下游**
- Backfill 期间告知下游团队数据可能不完整
- Backfill 完成后通知下游可以刷新

---
## 4. On-call Runbook

### Runbook 是什么？
针对特定故障的**标准操作程序（SOP）**，让任何 on-call 工程师（包括不熟悉该系统的人）能够快速处理事故。

### Runbook 模板
```markdown
# Runbook: Orders Pipeline 延迟告警

## 告警触发条件
- fct_orders 表数据延迟超过 60 分钟
- 告警来源：Datadog monitor `data.orders.freshness`

## 影响范围
- 实时销售 Dashboard 数据停止更新
- 运营团队无法查看当日订单

## 排查步骤

### Step 1: 确认问题
```sql
SELECT MAX(created_at) FROM analytics.fct_orders;
-- 若超过 1 小时前 → 确认问题
```

### Step 2: 检查上游
1. 打开 Airflow UI: https://airflow.company.com
2. 找到 `orders_pipeline` DAG
3. 查看最近运行状态

### Step 3: 分情况处理

**情况 A: DAG 失败**
1. 点击失败 Task → 查看 Logs
2. 常见错误处理：
   - `Connection timeout` → 重试（Clear task）
   - `Out of memory` → 联系 @data-platform-team
   - `Table not found` → 检查 schema 变更

**情况 B: DAG 运行中但很慢**
1. 检查 Spark UI: https://spark.company.com
2. 是否有数据倾斜？（Tasks 里某个特别慢）

**情况 C: DAG 未触发**
1. 检查 Airflow Scheduler 状态
2. 手动触发: `airflow dags trigger orders_pipeline`

## 升级条件
- 延迟超过 2 小时 → 升级 @data-lead
- 数据丢失 → 立即升级 @vp-engineering

## 联系方式
- Primary On-call: PagerDuty
- Data Team Slack: #data-incidents
- Runbook 更新人: @alice
```

### 优秀 Runbook 的特征
1. **可执行**：每步都有具体命令/链接，不依赖经验
2. **有预期结果**：执行完能知道是否成功
3. **有决策树**：不同情况有对应处理路径
4. **升级路径清晰**：什么情况升级，找谁
5. **保持更新**：每次处理后更新 Runbook

---
## 5. 练习题

### Q1 [高频] 描述你处理过的一次数据事故，从发现到解决的完整过程？

<details><summary>参考思路（STAR 法则）</summary>

**结构**：Situation（背景）→ Task（任务）→ Action（行动）→ Result（结果）

**示例**：
- **S**：某天早晨收到告警，销售 Dashboard 显示当日订单为 0
- **T**：需要在 9AM 业务会议前恢复数据（30 分钟内）
- **A**：
  1. 查 Airflow，发现 `fct_orders` DAG 成功了，但数据为空
  2. 检查上游，发现 `raw.orders` 分区存在，但行数只有昨天的 3%
  3. 追溯到 Kafka consumer，发现 consumer group lag 极高
  4. 发现是 Kafka broker 一个 partition 的 leader 选举失败，导致写入停顿了 2 小时
  5. Kafka 自动恢复后，重启 consumer，从上次 offset 继续消费
  6. 触发 `fct_orders` DAG 重跑当天分区
- **R**：45 分钟后数据恢复，业务会议推迟 15 分钟。事后加了 Kafka consumer lag 告警
</details>

---

### Q2 [高频] 如何设计幂等的数据 pipeline？

<details><summary>参考答案</summary>

**幂等**：同一个 pipeline 运行多次，结果与运行一次相同（不产生重复/不一致数据）。

**设计方法**：
1. **分区覆盖**：按日期分区，每次运行完整覆盖目标分区（不追加）
   ```sql
   INSERT OVERWRITE PARTITION (date='2024-01-15') ...
   ```
2. **UPSERT（Merge）**：用唯一键合并，有则更新无则插入
3. **先删后插**：在事务中清空目标范围后重新插入
4. **唯一键约束**：数据库层面防止重复插入（INSERT IGNORE 或 ON CONFLICT DO NOTHING）
5. **运行 ID 追踪**：每次运行写入 `pipeline_run_id`，重跑前先清除同 run_id 的数据

**测试幂等性**：运行 pipeline → 验证结果 → 再运行一次 → 结果应该完全一样。
</details>

---

### Q3 Backfill 时如何避免影响线上实时 pipeline？

<details><summary>参考答案</summary>

1. **限制并发**：`--max-active-runs 1`，一次只处理一个日期分区
2. **错峰执行**：安排在非业务高峰时段（深夜/周末）
3. **资源隔离**：
   - Spark：使用独立的 cluster 或降低 executor 数量
   - BigQuery：使用 reservation，给 backfill 和实时 pipeline 分配独立 slot
4. **优先级排队**：Airflow Pool 设置，实时 pipeline 优先级高于 backfill
5. **通知下游**：告知消费者 backfill 期间数据可能不完整，暂缓决策
6. **分批次执行**：每次只 backfill 最近 7 天，观察稳定后再扩大范围
</details>

---

### Q4 什么是好的 Post-mortem（事故复盘）？应该包含哪些内容？

<details><summary>参考答案</summary>

**原则**：Blameless（无责化），聚焦系统改进，不指责个人。

**核心内容**：
1. **摘要**：事故时间线、影响范围、严重程度
2. **时间线**：从发现到解决的完整时序
3. **根因分析**：5 Whys，找到真正的系统性原因
4. **影响评估**：受影响用户数、SLA 违反时长、数据损失
5. **改进措施**（最重要）：具体的、有 Owner、有 deadline 的行动项
   - 短期：修复 bug、更新 runbook
   - 长期：加监控、改架构、培训
6. **经验教训**：做得好的地方 + 需要改进的地方

**时间**：事故后 48-72 小时内完成，趁记忆清晰。
</details>

---

### Q5 [高频] 用 STAR 法则描述：你如何在没有停机的情况下修复了生产数据问题？

<details><summary>参考思路</summary>

**关键点**：
- 问题隔离：有问题的数据不暴露给消费者（写入 quarantine 表）
- 并行修复：修复逻辑在影子表/临时分区运行，验证正确后再切换
- 原子切换：用视图切换或分区 SWAP，而非逐行更新
- 保留回滚能力：Delta Lake Time Travel 或备份分区

**示例场景**：发现过去 3 天的收入计算错误（汇率应用错误），但报表不能停
1. 在影子表 `fct_orders_corrected` 用正确逻辑重算 3 天数据
2. 与财务团队验证数字
3. 用原子 `SWAP` 操作替换原表（Delta Lake 或 BigQuery 表 rename）
4. 整个过程用户无感知（服务未中断）
5. 通知 BI 刷新 Dashboard
</details>